In [1]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

texts = {
    "english": "A token is a small piece of text that a language model reads. One word can be split into several tokens. That is why text in languages like Hindi often costs more than English.",
    "hindi": "टोकन वह छोटा टुकड़ा होता है जिसे भाषा मॉडल पढ़ता है। एक शब्द कई टोकन में बँट सकता है। इसलिए हिंदी जैसे पाठ की लागत अक्सर अंग्रेज़ी से ज़्यादा होती है।",
    "code": "def add(a, b):\n    result = a + b\n    return result\n\nprint(add(2, 3))",
    "numbers": "3141592653 2718281828 1618033988 1414213562",
}

for name, text in texts.items():
    tokens = enc.encode(text)
    print(f"{name:8} | characters: {len(text):4} | tokens: {len(tokens):4}")

english  | characters:  176 | tokens:   37
hindi    | characters:  150 | tokens:  159
code     | characters:   69 | tokens:   25
numbers  | characters:   43 | tokens:   19


In [2]:
import sys
print(sys.executable)

d:\LLM\LLM_module_01\llm\Scripts\python.exe


In [2]:
import os, time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

PROVIDERS = {
    "groq": ("https://api.groq.com/openai/v1", "groq_api", "openai/gpt-oss-120b"),
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini_api_key", "gemini-3.8-flash"),
}

def run(provider, prompt, max_tokens=400):
    base_url, key_env, model = PROVIDERS[provider]
    client = OpenAI(base_url=base_url, api_key=os.environ[key_env])

    start = time.time()
    first = None
    usage = None
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        stream=True,
        stream_options={"include_usage": True},
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content and first is None:
            first = time.time() - start
        if getattr(chunk, "usage", None):
            usage = chunk.usage
    total = time.time() - start

    print(f"{provider} | time to first token: {first:.2f}s | total: {total:.2f}s")
    if usage:
        print(f"  input tokens: {usage.prompt_tokens} | output tokens: {usage.completion_tokens}")

run("groq", "Explain what a token is in 3 sentences.")
run("gemini", "Explain what a token is in 3 sentences.")

groq | time to first token: 2.15s | total: 2.38s
  input tokens: 81 | output tokens: 156
gemini | time to first token: 95.56s | total: 95.57s
  input tokens: 11 | output tokens: 13


In [3]:
def run(provider, prompt, max_tokens=400):
    base_url, key_env, model = PROVIDERS[provider]
    client = OpenAI(base_url=base_url, api_key=os.environ[key_env],
                    max_retries=0, timeout=60)

    start = time.time()
    first = None
    usage = None
    text = ""
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        stream=True,
        stream_options={"include_usage": True},
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            if first is None:
                first = time.time() - start
            text += chunk.choices[0].delta.content
        if getattr(chunk, "usage", None):
            usage = chunk.usage
    total = time.time() - start

    print(f"{provider} | first token: {first:.2f}s | total: {total:.2f}s")
    if usage:
        print(f"  input: {usage.prompt_tokens} | output: {usage.completion_tokens}")
    print("  answer:", text[:200])

# Groq has a generous limit, so repeat it 3 times
for _ in range(3):
    run("groq", "Explain what a token is in 3 sentences.")

groq | first token: 0.59s | total: 0.75s
  input: 81 | output: 123
  answer: A token is the smallest unit of text that a language model processes, which can be a single character, a whole word, or a common sub‑word piece (like “ing” or “un”). During tokenization, the input tex
groq | first token: 0.57s | total: 0.76s
  input: 81 | output: 160
  answer: A token is the smallest unit of text that a language model processes, which can be a single character, a whole word, or even a common sub‑word fragment like “ing”. When you input or generate text, the
groq | first token: 0.41s | total: 0.61s
  input: 81 | output: 135
  answer: A token is a discrete unit of text that a language model processes, typically representing a word, sub‑word, punctuation mark, or even a single character. During inference, the model reads a sequence 
